In [ ]:
!pip install torch torchvision albumentations opencv-python matplotlib huggingface_hub fiftyone -q

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
import os

# Download 5000 train + 500 val samples (no login needed)
train_dataset = foz.load_zoo_dataset(
    "bdd100k",
    split="train",
    label_types=["drivable"],
    max_samples=5000
)

val_dataset_fo = foz.load_zoo_dataset(
    "bdd100k",
    split="validation",
    label_types=["drivable"],
    max_samples=500
)

# Export to folders our BDDDataset class can read
train_dataset.export(
    export_dir="./data/train",
    dataset_type=fo.types.ImageSegmentationDirectory,
    label_field="drivable",
)

val_dataset_fo.export(
    export_dir="./data/val",
    dataset_type=fo.types.ImageSegmentationDirectory,
    label_field="drivable",
)

print("✅ Dataset ready!")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset_fo)}")

In [ ]:
import os, glob, time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def preprocess_mask(mask):
    out = np.zeros_like(mask)
    out[mask == 127] = 1
    out[mask == 191] = 2
    return out

class BDDDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_paths  = sorted(glob.glob(os.path.join(img_dir,  "*.jpg")))
        self.mask_paths = sorted(glob.glob(os.path.join(mask_dir, "*.png")))
        self.transform  = transform
    def __len__(self):
        return len(self.img_paths)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.img_paths[idx]), cv2.COLOR_BGR2RGB)
        mask = preprocess_mask(cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE))
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        return img, mask.long()

train_transform = A.Compose([
    A.Resize(256, 512), A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3), A.Rotate(limit=10, p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(256, 512),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()
])

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return skip, self.pool(skip)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = ConvBlock(out_ch * 2, out_ch)
    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))

class UNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.enc1 = EncoderBlock(3, 32);   self.enc2 = EncoderBlock(32, 64)
        self.enc3 = EncoderBlock(64, 128); self.enc4 = EncoderBlock(128, 256)
        self.bottleneck = ConvBlock(256, 512)
        self.dec4 = DecoderBlock(512, 256); self.dec3 = DecoderBlock(256, 128)
        self.dec2 = DecoderBlock(128, 64);  self.dec1 = DecoderBlock(64, 32)
        self.final = nn.Conv2d(32, num_classes, kernel_size=1)
    def forward(self, x):
        s1,x=self.enc1(x); s2,x=self.enc2(x); s3,x=self.enc3(x); s4,x=self.enc4(x)
        x=self.bottleneck(x)
        x=self.dec4(x,s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        return self.final(x)

class DiceLoss(nn.Module):
    def __init__(self, num_classes=3, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes; self.smooth = smooth
    def forward(self, preds, targets):
        preds = torch.softmax(preds, dim=1); loss = 0
        for c in range(self.num_classes):
            inter = (preds[:,c] * (targets==c).float()).sum()
            union = preds[:,c].sum() + (targets==c).float().sum()
            loss += 1 - (2*inter + self.smooth)/(union + self.smooth)
        return loss / self.num_classes

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(); self.dice = DiceLoss(num_classes=3)
    def forward(self, preds, targets):
        return self.ce(preds, targets) + self.dice(preds, targets)

def compute_miou(preds, targets, num_classes=3):
    preds = torch.argmax(preds, dim=1); iou_list = []
    for c in range(num_classes):
        inter = ((preds==c)&(targets==c)).sum().float()
        union = ((preds==c)|(targets==c)).sum().float()
        if union > 0: iou_list.append((inter/union).item())
    return sum(iou_list)/len(iou_list) if iou_list else 0.0

In [ ]:
train_ds = BDDDataset("./data/train/data", "./data/train/labels", transform=train_transform)
val_ds   = BDDDataset("./data/val/data",   "./data/val/labels",   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} samples | Val: {len(val_ds)} samples")

In [ ]:
model     = UNet(num_classes=3).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
criterion = CombinedLoss()
scaler    = GradScaler()

EPOCHS    = 25
best_miou = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with autocast():
            out  = model(imgs)
            loss = criterion(out, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    model.eval()
    miou_scores = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            miou_scores.append(compute_miou(out, masks))

    epoch_miou = sum(miou_scores) / len(miou_scores)
    avg_loss   = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f} | mIoU: {epoch_miou:.4f}")

    if epoch_miou > best_miou:
        best_miou = epoch_miou
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✅ Saved best model (mIoU: {best_miou:.4f})")

    scheduler.step()

print(f"\n🎉 Training complete! Best mIoU: {best_miou:.4f}")

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

dummy = torch.randn(1, 3, 256, 512).to(device)
for _ in range(10): model(dummy)  # warmup

start = time.time()
RUNS  = 100
for _ in range(RUNS):
    with torch.no_grad(): model(dummy)
fps = RUNS / (time.time() - start)
print(f"✅ FPS: {fps:.1f} | Inference time: {1000/fps:.2f} ms/frame")

In [ ]:
import random

COLOR_MAP = {0: [0,0,0], 1: [0,255,0], 2: [255,165,0]}

def mask_to_color(mask):
    h, w  = mask.shape
    color = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, col in COLOR_MAP.items():
        color[mask == cls] = col
    return color

model.eval()
fig, axes = plt.subplots(5, 3, figsize=(15, 20))
axes[0][0].set_title("Input Image")
axes[0][1].set_title("Ground Truth")
axes[0][2].set_title("Prediction")

indices = random.sample(range(len(val_ds)), 5)
for row, idx in enumerate(indices):
    img, mask = val_ds[idx]s
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()

    img_np = img.permute(1, 2, 0).numpy()
    img_np = (img_np * [0.229,0.224,0.225] + [0.485,0.456,0.406]).clip(0, 1)

    axes[row][0].imshow(img_np)
    axes[row][1].imshow(mask_to_color(mask.numpy()))
    axes[row][2].imshow(mask_to_color(pred))
    for ax in axes[row]: ax.axis("off")

plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150)
plt.show()
print("✅ Saved sample_predictions.png")